# MoneyAI 💰

Agente de contabilidad personal que analiza tus movimientos bancarios desde Gmail.


In [56]:
import os.path
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# Scopes para acceso de solo lectura a Gmail
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

def get_gmail_credentials():
    """Obtiene las credenciales de Gmail, solicitando autorización si es necesario."""
    creds = None
    
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    
    return creds

# Autenticar con Gmail
creds = get_gmail_credentials()
service = build('gmail', 'v1', credentials=creds)

print("✅ Conectado a Gmail correctamente")

✅ Conectado a Gmail correctamente


In [57]:
import base64
import re

def extraer_monto(texto):
    """Extrae el monto de un texto usando regex."""
    # Patrones para diferentes formatos de montos
    patrones = [
        r'Monto\s*\$?\s*([\d.,]+)',  # Monto $1.234,56 o Monto 1234
        r'Monto\s+U\$S\s*([\d.,]+)',  # Monto U$S 14,16
        r'\$\s*([\d.,]+)',  # $1.234,56
        r'U\$S\s*([\d.,]+)',  # U$S 14,16
        r'ARS\s*([\d.,]+)',  # ARS 1234
        r'USD\s*([\d.,]+)',  # USD 1234
    ]
    
    for patron in patrones:
        match = re.search(patron, texto, re.IGNORECASE)
        if match:
            monto = match.group(1)
            # Determinar moneda
            if 'U$S' in texto.upper() or 'USD' in texto.upper():
                return f"U$S {monto}"
            else:
                return f"${monto}"
    return None

def limpiar_html(html_text):
    """Limpia tags HTML y devuelve texto plano."""
    texto = re.sub(r'<[^>]+>', ' ', html_text)
    texto = texto.replace('&nbsp;', ' ').replace('&amp;', '&')
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def extraer_cuerpo_email(payload):
    """Extrae el cuerpo de texto del email recursivamente."""
    texto_plano = ""
    html = ""
    
    if 'body' in payload and payload['body'].get('data'):
        data = payload['body']['data']
        contenido = base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
        mime_type = payload.get('mimeType', '')
        if 'html' in mime_type:
            html = contenido
        else:
            texto_plano = contenido
    
    if 'parts' in payload:
        for part in payload['parts']:
            mime_type = part.get('mimeType', '')
            if part['body'].get('data'):
                data = part['body']['data']
                contenido = base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
                if mime_type == 'text/plain':
                    texto_plano = contenido
                elif mime_type == 'text/html':
                    html = contenido
            elif mime_type.startswith('multipart/'):
                resultado = extraer_cuerpo_email(part)
                if resultado:
                    return resultado
    
    if texto_plano.strip():
        return texto_plano
    elif html:
        return limpiar_html(html)
    return ""

def buscar_emails(query, max_results=50):
    """Busca emails en Gmail según una query y devuelve lista de emails procesados."""
    results = service.users().messages().list(userId='me', q=query, maxResults=max_results).execute()
    messages = results.get('messages', [])
    
    emails = []
    for msg in messages:
        message = service.users().messages().get(userId='me', id=msg['id'], format='full').execute()
        headers = message['payload'].get('headers', [])
        
        # Extraer contenido y monto
        contenido = extraer_cuerpo_email(message['payload'])
        snippet = message.get('snippet', '')
        
        # Buscar monto en contenido o snippet
        monto = extraer_monto(contenido) or extraer_monto(snippet)
        
        emails.append({
            'id': msg['id'],
            'subject': next((h['value'] for h in headers if h['name'] == 'Subject'), 'Sin asunto'),
            'from': next((h['value'] for h in headers if h['name'] == 'From'), 'Desconocido'),
            'date': next((h['value'] for h in headers if h['name'] == 'Date'), 'Sin fecha'),
            'snippet': snippet,
            'contenido': contenido[:1000] if contenido else snippet,
            'monto_extraido': monto,  # ✅ Monto extraído directamente
            'payload': message['payload']
        })
    
    return emails

def mostrar_emails(emails):
    """Muestra los emails de forma legible."""
    print(f"📧 Se encontraron {len(emails)} emails\n")
    for i, email in enumerate(emails):
        print(f"--- Email {i+1} ---")
        print(f"📌 Asunto: {email['subject']}")
        print(f"👤 De: {email['from']}")
        print(f"📅 Fecha: {email['date']}")
        print(f"📝 Resumen: {email['snippet'][:150]}...")
        print()


In [58]:
# Buscar avisos de Santander
santander_emails = buscar_emails('from:mensajesyavisos@mails.santander.com.ar')
mostrar_emails(santander_emails)


📧 Se encontraron 50 emails

--- Email 1 ---
📌 Asunto: Se realizó un débito en tu cuenta
👤 De: Aviso Santander <mensajesyavisos@mails.santander.com.ar>
📅 Fecha: Sat, 3 Jan 2026 11:21:41 -0300 (ART)
📝 Resumen: Información sobre el débito en tu cuenta por recurrencia Hola GARCIA BARRIOLA LEANDRO OMAR NICOLAS, Queremos comunicarte que se realizó un débito en t...

--- Email 2 ---
📌 Asunto: Aviso de transferencia
👤 De: Aviso Santander <mensajesyavisos@mails.santander.com.ar>
📅 Fecha: Fri, 2 Jan 2026 17:05:39 -0300 (ART)
📝 Resumen: Información sobre tu transferencia Se realizó la siguiente transferencia a tu nombre: Destinatario 20348750454 Cuenta de origen Cuenta en Pesos XXX-XX...

--- Email 3 ---
📌 Asunto: Aviso de transferencia
👤 De: Aviso Santander <mensajesyavisos@mails.santander.com.ar>
📅 Fecha: Fri, 2 Jan 2026 10:17:05 -0300 (ART)
📝 Resumen: Información sobre tu transferencia Se realizó la siguiente transferencia a tu nombre: Destinatario 20922973424 Cuenta de origen Cuenta en Pesos 

In [59]:
# DEBUG: Ver el contenido extraído de un email
print("📧 Contenido extraído del primer email:")
print("=" * 60)
print(santander_emails[0].get('contenido', 'NO HAY CONTENIDO'))
print("=" * 60)
print(f"\n📝 Snippet original: {santander_emails[0]['snippet'][:200]}...")


📧 Contenido extraído del primer email:
Se realizó un débito en tu cuenta a, body, img, p, table, tbody, td, thead, tr { padding: 0; margin: 0; border: none; border-spacing: 0; border-collapse: collapse; vertical-align: middle; color: #767676; font-family: 'Helvetica', 'Arial', sans-serif; } .wrapper { padding-left: 10px; padding-right: 10px; } .border { margin-top: 24px; border: 1px solid #cccccc; } p { padding: 24px; line-height: 1.6; } h1 { font-size: 32px; font-weight: normal; color: #EC0000; margin: 24px 0 0 24px; } .item-data, .item-title { width: 252px; border-bottom: 1px solid #767676; font-size: 16px; } .last { border: none; } .item-title { text-align: left; } .item-data { font-weight: bold; text-align: right; } .legal-text { font-size: 14px; line-height: 1.4; color: #818181; } .legal-text b { color: #444444; } .legal-text .red { color: #EC0000; } .header-img { background: url(https://wap.santander.com.ar/mensaje/header_step_celeste.png ) no-repeat; background-size: cover; back

In [60]:
import os
from typing import TypedDict, Literal
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Cargar variables de entorno desde .env
load_dotenv()

# Verificar que existe la API key
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("⚠️ Falta OPENAI_API_KEY en el archivo .env")

# Inicializar el modelo (gpt-4o-mini es rápido y económico)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ Modelo OpenAI (gpt-4o-mini) configurado correctamente")


✅ Modelo OpenAI (gpt-4o-mini) configurado correctamente


In [61]:
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum

# Definir las categorías
class CategoriaEmail(str, Enum):
    TRANSFERENCIA_MP_COMIDAS = "transferencia_mp_comidas"
    DEBITO_CUENTA = "debito_en_cuenta"
    DEBITO_TARJETA_CREDITO = "debito_pago_tarjeta_credito"
    DEBITO_PRESTAMO = "debito_automatico_prestamo"
    AYUDA_PAPAS = "ayuda_papas"
    OTRO = "otro"

# Schema para la respuesta estructurada
class EmailCategorizado(BaseModel):
    categoria: CategoriaEmail = Field(description="Categoría del email bancario")
    monto: Optional[str] = Field(default=None, description="Monto de la transacción si está disponible")
    descripcion: str = Field(description="Breve descripción de la transacción")

# Crear el modelo con salida estructurada
llm_categorizer = llm.with_structured_output(EmailCategorizado)

def categorizar_email(email: dict) -> dict:
    """Categoriza un email bancario usando el LLM."""
    
    prompt = f"""Analiza este email bancario de Santander y categorízalo.

CATEGORÍAS DISPONIBLES:
- transferencia_mp_comidas: Transferencias a MercadoPago para comidas. CRITERIOS: Servicio = "FONDEO" Y cuenta destino contiene "0077" (ARS XXX-XXX0077)
- debito_en_cuenta: Débitos directos en cuenta (no tarjeta, no préstamo)
- debito_pago_tarjeta_credito: Pagos con tarjeta de crédito (AMEX o Visa)
- debito_automatico_prestamo: Débitos automáticos de cuotas de préstamos
- ayuda_papas: Transferencias a MercadoPago para ayudar a los papás. CRITERIOS: Destinatario = "20922973424"
- otro: Cualquier otro tipo (promociones, avisos informativos, transferencias que NO cumplan los criterios anteriores, etc.)

IMPORTANTE: Para "transferencia_mp_comidas" AMBOS criterios deben cumplirse (FONDEO + cuenta 0077).

EMAIL A ANALIZAR:
Asunto: {email['subject']}
Contenido: {email.get('contenido', email['snippet'])}

Categoriza este email y extrae el monto si está disponible."""

    resultado = llm_categorizer.invoke(prompt)
    
    return {
        **email,
        'categoria': resultado.categoria.value,
        'monto': email.get('monto_extraido'),  # Usar el monto extraído por regex
        'descripcion_ia': resultado.descripcion
    }

print("✅ Función de categorización lista")


✅ Función de categorización lista


In [62]:
# Definir el estado del agente
class AgentState(TypedDict):
    emails_pendientes: List[dict]
    emails_categorizados: List[dict]
    email_actual: Optional[dict]
    indice: int

# Nodos del grafo
def inicializar(state: AgentState) -> AgentState:
    """Inicializa el procesamiento."""
    return {
        **state,
        "indice": 0,
        "emails_categorizados": []
    }

def seleccionar_email(state: AgentState) -> AgentState:
    """Selecciona el siguiente email a procesar."""
    idx = state["indice"]
    if idx < len(state["emails_pendientes"]):
        return {
            **state,
            "email_actual": state["emails_pendientes"][idx]
        }
    return {**state, "email_actual": None}

def procesar_email(state: AgentState) -> AgentState:
    """Procesa y categoriza el email actual."""
    email = state["email_actual"]
    if email:
        print(f"  📧 Procesando: {email['subject'][:50]}...")
        email_categorizado = categorizar_email(email)
        return {
            **state,
            "emails_categorizados": state["emails_categorizados"] + [email_categorizado],
            "indice": state["indice"] + 1
        }
    return state

def hay_mas_emails(state: AgentState) -> str:
    """Determina si hay más emails por procesar."""
    if state["indice"] < len(state["emails_pendientes"]):
        return "seleccionar"
    return END

# Construir el grafo
workflow = StateGraph(AgentState)

# Agregar nodos
workflow.add_node("inicializar", inicializar)
workflow.add_node("seleccionar", seleccionar_email)
workflow.add_node("procesar", procesar_email)

# Definir flujo
workflow.set_entry_point("inicializar")
workflow.add_edge("inicializar", "seleccionar")
workflow.add_edge("seleccionar", "procesar")
workflow.add_conditional_edges("procesar", hay_mas_emails)

# Compilar el agente
agente_categorizador = workflow.compile()

print("✅ Agente de categorización compilado")


✅ Agente de categorización compilado


In [63]:
# Ejecutar el agente con los emails de Santander
import time

# Cantidad de emails a procesar (reducir si hay errores de cuota)
CANTIDAD_EMAILS = 5

print(f"🚀 Iniciando categorización de {CANTIDAD_EMAILS} emails...\n")

resultado = agente_categorizador.invoke({
    "emails_pendientes": santander_emails[:CANTIDAD_EMAILS],
    "emails_categorizados": [],
    "email_actual": None,
    "indice": 0
})

print("\n✅ Categorización completada!")
print(f"📊 Total procesados: {len(resultado['emails_categorizados'])}")


🚀 Iniciando categorización de 5 emails...

  📧 Procesando: Se realizó un débito en tu cuenta...


  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de transferencia...
  📧 Procesando: Aviso de alta de destinatario de transferencia...
  📧 Procesando: Aviso de transferencia...

✅ Categorización completada!
📊 Total procesados: 5


In [64]:
# Mostrar resultados agrupados por categoría
from collections import defaultdict

emails_por_categoria = defaultdict(list)
for email in resultado['emails_categorizados']:
    emails_por_categoria[email['categoria']].append(email)

# Emojis por categoría
emojis = {
    'transferencia_mp_comidas': '🍔',
    'debito_en_cuenta': '🏦',
    'debito_pago_tarjeta_credito': '💳',
    'debito_automatico_prestamo': '📋',
    'ayuda_papas': '👨‍👩‍👧‍👦',
    'otro': '📌'
}

# Mostrar resumen
print("=" * 60)
print("📊 RESUMEN DE EMAILS CATEGORIZADOS")
print("=" * 60)

for categoria, emails in emails_por_categoria.items():
    emoji = emojis.get(categoria, '📧')
    print(f"\n{emoji} {categoria.upper().replace('_', ' ')} ({len(emails)} emails)")
    print("-" * 40)
    for email in emails:
        monto = f" - {email['monto']}" if email.get('monto') else ""
        print(f"  • {email['subject'][:45]}...{monto}")
        print(f"    └─ {email['descripcion_ia'][:60]}...")

print("\n" + "=" * 60)


📊 RESUMEN DE EMAILS CATEGORIZADOS

🏦 DEBITO EN CUENTA (1 emails)
----------------------------------------
  • Se realizó un débito en tu cuenta... - $150000.0
    └─ Se realizó un débito en tu cuenta...

📌 OTRO (4 emails)
----------------------------------------
  • Aviso de transferencia... - $1.500.000,00
    └─ Aviso de transferencia confirmada...
  • Aviso de transferencia... - $330.000,00
    └─ Aviso de transferencia confirmada...
  • Aviso de alta de destinatario de transferenci...
    └─ Aviso de alta de nuevo destinatario de transferencias...
  • Aviso de transferencia... - $40.000,00
    └─ Aviso de transferencia confirmada...

